# LiveScene on Google Colab

Runs the full-quality profile: a large open-vocabulary detector, a large depth model and a 3B or 7B
vision-language model, chosen automatically from the GPU memory Colab gives you. Then it measures
hallucination (POPE, CHAIR) and detection accuracy (COCO mAP).

Before running: **Runtime > Change runtime type > GPU**. A T4 runs the `colab-t4` profile; an L4 or
A100 runs the larger `colab` profile.

The live webcam mode is for a local machine; see the README.

In [ ]:
# 1. Confirm a GPU is attached
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
print(torch.__version__, torch.cuda.get_device_name(0),
      round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GB')

In [ ]:
# 2. Get the code
import glob, os, zipfile

USE_ZIP = False  # set True while the GitHub repository is private

if not os.path.exists('/content/livescene'):
    if USE_ZIP:
        from google.colab import files
        up = files.upload()  # the ZIP from GitHub: Code > Download ZIP
        zipfile.ZipFile(next(iter(up))).extractall('/content')
        os.rename(next(d for d in glob.glob('/content/livescene-*') if os.path.isdir(d)), '/content/livescene')
    else:
        !git clone -q https://github.com/HameesSajid/livescene.git /content/livescene
%cd /content/livescene

In [ ]:
# 3. Install the libraries (PyTorch with CUDA is already on Colab)
!pip -q install -r requirements.txt
!pip -q install https://github.com/ultralytics/CLIP/archive/refs/heads/main.zip

In [ ]:
# 4. Unit tests (no GPU needed) and the profile that will be used
!python tests/test_memory.py && python tests/test_describe.py && python tests/test_chair.py
from livescene.config import pick
cfg = pick('auto')
VLM, DETECTOR, SIDE = cfg.vlm, cfg.verifier, cfg.vlm_side
print(cfg)

In [ ]:
# 5. Upload a video, run the pipeline on it, and ask it a question
from google.colab import files
video = next(iter(files.upload()))
QUESTION = "What is happening in this video?"
!python run.py --source "$video" --headless --save outputs/colab_demo.mp4 --report outputs/colab_bench.json --ask "$QUESTION" --ask-at 3

In [ ]:
# 6. Watch the annotated result (re-encoded to H.264 so the browser can play it)
!ffmpeg -y -loglevel error -i outputs/colab_demo.mp4 -vcodec libx264 outputs/colab_demo_h264.mp4
from IPython.display import Video
Video('outputs/colab_demo_h264.mp4', embed=True, width=800)

In [ ]:
# 7. Latency per stage, then descriptions, answers and events
import json
rep = json.load(open('outputs/colab_bench.json'))
print(rep['profile'], '|', rep['device'], '|', round(rep['fps'], 1), 'FPS |', round(rep['peak_vram_mb']), 'MB peak GPU memory')
for name, s in rep['latency'].items():
    print(f"{name:25s} median {s['p50_ms']:8.1f} ms   p95 {s['p95_ms']:8.1f} ms")
print()
for e in rep['log']:
    if e['kind'] in ('description', 'answer', 'rejected'):
        print(f"{e['t']:7.2f}s  {e['kind']:11s} {e['text']}")

In [ ]:
# 8. Hallucination benchmarks for the language model this GPU selected
#    POPE: yes/no object probes.  CHAIR: objects named in captions that are not in the image.
!python tools/eval_hallucination.py --vlm "$VLM" --task pope --mode plain --images 200 --max-side $SIDE --out outputs/pope_colab_plain.json
!python tools/eval_hallucination.py --vlm "$VLM" --task chair --mode plain --images 200 --tokens 120 --max-side $SIDE --out outputs/chair_colab_plain.json
!python tools/eval_hallucination.py --vlm "$VLM" --task chair --mode verified --detector $DETECTOR --images 200 --tokens 120 --max-side $SIDE --out outputs/chair_colab_verified.json

In [ ]:
# 9. Detection accuracy on the full COCO val2017 set (5,000 images; downloads about 1 GB)
!python tools/eval_coco.py --n 5000 --weights $DETECTOR --out outputs/coco_colab_full.json

In [ ]:
# 10. Download results before the session ends (Colab deletes its files on disconnect)
import glob
from google.colab import files
for path in ['outputs/colab_demo_h264.mp4'] + sorted(glob.glob('outputs/*colab*.json')):
    if os.path.exists(path):
        files.download(path)